In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

I renamed the files to have `_downloaded` suffix

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system.
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If needed, install the Python client with:

```bash
pip install ollama
```


## Asking meaningful questions without tools

In [5]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

The FAQ doesn’t mention **Olama** specifically.

If you mean running the course locally, the FAQ says you can run it locally instead of Codespaces if you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you do run locally, you should document your setup and keep the environment reproducible.


In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — but I’d need a bit more context to answer accurately.\n\nPlease tell me:\n- the course name\n- where it’s offered\n- whether it has already started\n- whether you mean joining as a student, guest, or late enrolment\n\nIf you want, I can also help you draft a quick message to the instructor or course admin asking if late registration is still possible.'

In [7]:
messages = [
    {"role": "user", "content": "How do I run Olama locally?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'If you mean **Ollama** (often misspelled “Olama”), the easiest way to run it locally is:\n\n## 1) Install Ollama\n### macOS\nDownload from: https://ollama.com/download\n\n### Linux\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### Windows\nDownload the Windows installer from: https://ollama.com/download\n\n---\n\n## 2) Start the Ollama service\nUsually it starts automatically after install. If needed:\n\n```bash\nollama serve\n```\n\n---\n\n## 3) Run a model locally\nFor example, to download and run Llama 3:\n\n```bash\nollama run llama3\n```\n\nOther examples:\n```bash\nollama run mistral\nollama run phi3\nollama run gemma\n```\n\n---\n\n## 4) Use it from the command line\nOnce running, you can chat directly in the terminal:\n\n```bash\nollama run llama3\n```\n\n---\n\n## 5) Use the local API\nOllama exposes a local API on:\n\n```bash\nhttp://localhost:11434\n```\n\nExample request:\n\n```bash\ncurl http://localhost:11434/api/generate -d \'{\n  "model": "llama3",\n 

## Define the search function that will have an agentic element to it

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
# tell the model about the search tool, will be sending .json to OpenAI
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
# sending the question with the tool
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    # tools addition
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Olama locally run install local server Ollama FAQ"}', call_id='call_Bav3e6oEInUOvZRIdGGB9qtR', name='search', type='function_call', id='fc_00c738719c377b45006a36a4d6ad7c81a38ddb517b74a73f13', namespace=None, status='completed')]

One this that's interesting to note is that the query that was sent to OpenAI is not the originial query of , ""How do I run Olama locally?" but rather "un Ollama locally install start model pull command FAQ", which sound like gibberish to us, but makes more sense to machines

In [11]:
len(response.output)

1

In [12]:
# we know the arguments (from ResponseFunctionToolCall) and now we need to parse them
import json
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
# turn the results back to json to send back to LLM
result_json = json.dumps(results, indent=2)

## the LLM said, "I want to invoke the search function with these arguments (query)"

In [13]:
args

{'query': 'Olama locally run install local server Ollama FAQ'}

In [14]:
call.name

'search'

In [15]:
# result_json is what we need to send back to the LLM
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [16]:
# append the function call output to messages, which is the message history
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [17]:
messages

[{'role': 'user', 'content': 'How do I run Olama locally?'},
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run install local server Ollama FAQ"}', call_id='call_Bav3e6oEInUOvZRIdGGB9qtR', name='search', type='function_call', id='fc_00c738719c377b45006a36a4d6ad7c81a38ddb517b74a73f13', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Bav3e6oEInUOvZRIdGGB9qtR',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a terminal and

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    # now messages contains the previously sent use question and response
    input=messages,
    # tools addition
    tools=[search_tool],
)


In [20]:
print(response.output_text)

To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install from https://ollama.com/download
   - **Windows**: download the `.msi` from https://ollama.com/download
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a JSON response listing models.

4. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you want, I can also show you how to run Ollama in the background or connect it to a RAG app.


In [ ]:
# But what happens when an LLM wants to make multiple calls instead of a singular call to tools? 